In [4]:
# filter_trials_by_codes.py
"""
Filter trials to only include those with entity codes present in your MIMIC data.
"""

import json
import os
import pandas as pd
from config import Config

def load_mimic_codes(cfg):
    """Load all diagnosis, medication, and lab codes from MIMIC data."""
    diag_path = f"{cfg.OUTPUT_DIR}/diagnoses_clean.parquet"
    rx_path = f"{cfg.OUTPUT_DIR}/prescriptions_clean.parquet"
    labs_path = f"{cfg.OUTPUT_DIR}/labs_clean.parquet"
    
    diag_df = pd.read_parquet(diag_path)
    rx_df = pd.read_parquet(rx_path)
    labs_df = pd.read_parquet(labs_path)
    
    diagnosis_codes = set(diag_df['ICD10_CODE'].astype(str).unique())
    medication_codes = set(rx_df['NDC'].astype(str).unique())
    lab_codes = set(labs_df['ITEMID'].astype(str).unique())
    
    print(f"📊 MIMIC Codes:")
    print(f"   Diagnosis: {len(diagnosis_codes):,}")
    print(f"   Medication: {len(medication_codes):,}")
    print(f"   Lab: {len(lab_codes):,}")
    
    return diagnosis_codes, medication_codes, lab_codes

def find_trial_files():
    """Find trial files in various possible locations."""
    possible_paths = [
        "data/1000_trials/structured_clinical_trials.json",
        "structured_clinical_trials.json",
        "../data/1000_trials/structured_clinical_trials.json",
        "extracting_trials/structured_clinical_trials.json",
        "ctg-studies_1000.json",
    ]
    
    found_paths = []
    for path in possible_paths:
        if os.path.exists(path):
            found_paths.append(path)
            print(f"✅ Found trial file: {path}")
    
    return found_paths

def load_trials_from_file(filepath):
    """Load trials from a JSON file."""
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
        if isinstance(data, list):
            return data
        else:
            return [data]
    except Exception as e:
        print(f"⚠️ Error loading {filepath}: {e}")
        return []

def filter_trials_by_codes(trials, diagnosis_codes, medication_codes, lab_codes):
    """
    Filter trials to only those where at least one criterion matches existing codes.
    """
    all_codes = diagnosis_codes | medication_codes | lab_codes
    
    filtered_trials = []
    matched_criteria_count = 0
    total_criteria_count = 0
    
    for trial in trials:
        criteria = trial.get('criteria', [])
        total_criteria_count += len(criteria)
        
        # Check if any criterion has a code that exists in MIMIC
        has_match = False
        for c in criteria:
            code = c.get('entity_code', '')
            if code in all_codes:
                has_match = True
                matched_criteria_count += 1
        
        if has_match:
            filtered_trials.append(trial)
    
    print(f"\n📊 Filter Results:")
    print(f"   Total trials: {len(trials)}")
    print(f"   Trials with matches: {len(filtered_trials)}")
    print(f"   Total criteria: {total_criteria_count}")
    print(f"   Criteria with matching codes: {matched_criteria_count}")
    
    if total_criteria_count > 0:
        print(f"   Match rate: {matched_criteria_count/total_criteria_count*100:.2f}%")
    else:
        print(f"   Match rate: N/A (no criteria found)")
    
    return filtered_trials

def analyze_missing_codes(trials, all_codes):
    """Analyze which codes are missing from MIMIC."""
    missing_codes = {}
    
    for trial in trials:
        for c in trial.get('criteria', []):
            code = c.get('entity_code', '')
            if code and code not in all_codes:
                if code not in missing_codes:
                    missing_codes[code] = []
                missing_codes[code].append(trial.get('nct_id', 'Unknown'))
    
    if missing_codes:
        print("\n🔍 Top 10 Missing Codes:")
        sorted_missing = sorted(missing_codes.items(), key=lambda x: len(x[1]), reverse=True)
        for code, trial_list in sorted_missing[:10]:
            print(f"   {code}: appears in {len(trial_list)} trials")
    else:
        print("\n✅ All codes found in MIMIC data!")
    
    return missing_codes

def get_matching_codes(trials, all_codes):
    """Get codes that actually match."""
    matching_codes = set()
    for trial in trials:
        for c in trial.get('criteria', []):
            code = c.get('entity_code', '')
            if code in all_codes:
                matching_codes.add(code)
    
    return matching_codes

def main():
    cfg = Config()
    
    # 1. Load MIMIC codes
    diagnosis_codes, medication_codes, lab_codes = load_mimic_codes(cfg)
    all_codes = diagnosis_codes | medication_codes | lab_codes
    print(f"   Total unique codes: {len(all_codes):,}")
    
    # 2. Find and load trials
    print("\n🔍 Looking for trial files...")
    trial_files = find_trial_files()
    
    all_trials = []
    for filepath in trial_files:
        trials = load_trials_from_file(filepath)
        if trials:
            all_trials.extend(trials)
            print(f"   Loaded {len(trials)} trials from {filepath}")
    
    if not all_trials:
        print("\n❌ No trial files found!")
        print("\n💡 Please ensure your trial files are in one of these locations:")
        print("   - data/1000_trials/structured_clinical_trials.json")
        print("   - structured_clinical_trials.json")
        print("   - extracting_trials/structured_clinical_trials.json")
        print("   - ctg-studies_1000.json")
        return
    
    print(f"\n📂 Total trials loaded: {len(all_trials)}")
    
    # 3. Show sample of trial codes
    print("\n📋 Sample trial codes (first 3 trials):")
    for i, trial in enumerate(all_trials[:3]):
        print(f"   Trial {i+1}: {trial.get('nct_id', 'Unknown')}")
        criteria = trial.get('criteria', [])[:3]
        for c in criteria:
            print(f"      - {c.get('entity_code', 'N/A')} ({c.get('entity_type', 'N/A')})")
    
    # 4. Filter
    filtered = filter_trials_by_codes(all_trials, diagnosis_codes, medication_codes, lab_codes)
    
    if not filtered:
        print("\n❌ NO TRIALS MATCH! You need to download different trials.")
        print("\n💡 Recommended: Download trials for these conditions:")
        print("   heart failure, myocardial infarction, diabetes, pneumonia,")
        print("   sepsis, atrial fibrillation, hypertension, chronic kidney disease")
        
        # Analyze missing codes
        missing = analyze_missing_codes(all_trials, all_codes)
        
        # Show sample of MIMIC codes for comparison
        print("\n📋 Sample of MIMIC codes for reference:")
        sample_codes = list(all_codes)[:10]
        for code in sample_codes:
            print(f"   {code}")
        return
    
    # 5. Show matching codes
    matching_codes = get_matching_codes(filtered, all_codes)
    print(f"\n✅ Found {len(matching_codes)} matching codes in {len(filtered)} trials")
    
    # 6. Split back into train/eval
    split_idx = int(len(filtered) * 0.8)
    train_filtered = filtered[:split_idx]
    eval_filtered = filtered[split_idx:]
    
    # 7. Save filtered trials
    os.makedirs(cfg.TRIALS_DATA_DIR, exist_ok=True)
    
    train_path = f"{cfg.TRIALS_DATA_DIR}/structured_clinical_trials.json"
    eval_path = f"{cfg.TRIALS_DATA_DIR}/structured_clinical_trials_eval.json"
    
    with open(train_path, 'w') as f:
        json.dump(train_filtered, f, indent=2)
    print(f"\n✅ Saved {len(train_filtered)} training trials to {train_path}")
    
    with open(eval_path, 'w') as f:
        json.dump(eval_filtered, f, indent=2)
    print(f"✅ Saved {len(eval_filtered)} evaluation trials to {eval_path}")
    
    # 8. Also save to current directory for backward compatibility
    with open('structured_clinical_trials.json', 'w') as f:
        json.dump(train_filtered, f, indent=2)
    with open('structured_clinical_trials_eval.json', 'w') as f:
        json.dump(eval_filtered, f, indent=2)
    print(f"✅ Also saved to current directory")

if __name__ == "__main__":
    main()

📊 MIMIC Codes:
   Diagnosis: 4,229
   Medication: 437
   Lab: 178
   Total unique codes: 4,844

🔍 Looking for trial files...
✅ Found trial file: data/1000_trials/structured_clinical_trials.json
✅ Found trial file: structured_clinical_trials.json
   Loaded 149 trials from data/1000_trials/structured_clinical_trials.json
   Loaded 149 trials from structured_clinical_trials.json

📂 Total trials loaded: 298

📋 Sample trial codes (first 3 trials):
   Trial 1: NCT07732361
      - CODE_8730 (diagnosis)
      - CODE_8239 (diagnosis)
      - CODE_4891 (diagnosis)
   Trial 2: NCT07732309
      - CODE_1325 (diagnosis)
      - CODE_8108 (diagnosis)
      - CODE_6382 (diagnosis)
   Trial 3: NCT07732283
      - CODE_7930 (diagnosis)
      - CODE_9738 (diagnosis)
      - CODE_9913 (diagnosis)

📊 Filter Results:
   Total trials: 298
   Trials with matches: 0
   Total criteria: 4124
   Criteria with matching codes: 0
   Match rate: 0.00%

❌ NO TRIALS MATCH! You need to download different trials.

💡 Rec